# Day 31: Google ADK Deep Dive & Gemini

**Week 5 — Agent Development**

---

## 📚 Theory: The Google Ecosystem

If you are developing the Google Agent Development Kit (ADK), your primary underlying foundation model is going to be **Gemini**. Gemini has specific architectural advantages over OpenAI/Anthropic that your ADK must leverage.

### 1. Multimodality Natively
Unlike GPT-4 which bolts on a vision model, Gemini was trained from the ground up on text, images, audio, and video simultaneously. Your Agent Orchestrator doesn't need to use a tool to transcribe a video; you can pass the raw `.mp4` bytes directly into the context window.

### 2. Massive Context Windows & Context Caching
Gemini 1.5 Pro supports 2,000,000 tokens. However, sending 2M tokens on every step of an agent loop is incredibly expensive and slow. 
Google provides **Context Caching**. If you load a 1,000-page codebase into the prompt, you can *cache* it on Google's servers. Subsequent agent loops only pay a fraction of the cost and return results in milliseconds because the prefix of the prompt is already computed.

### 3. Structured Outputs (JSON Mode)
Agents rely on structured output to parse tool arguments. The ADK must enforce JSON schema constraints at the API level so the LLM is physically incapable of outputting malformed JSON.

## 🔨 Exercise 1: Architecting a Gemini ADK Wrapper

Let's design a Python class `GoogleAgent` that abstracts away the complexity of raw HTTP calls and implements Context Caching and Tool calling.

**Task**: Complete the skeleton class below to show how you would structure the initialization and the `chat()` method.

In [ ]:
class GoogleAgent:
    def __init__(self, system_instruction: str, tools: list = None, enable_caching: bool = False):
        """
        Initialize the Agent.
        """
        self.system_instruction = system_instruction
        self.tools = tools if tools else []
        self.history = []
        self.cache_id = None
        self.enable_caching = enable_caching
        
        # YOUR CODE HERE
        # If caching is enabled, simulate generating a cache ID for the system instruction.
        pass
        
    def chat(self, user_message: str) -> str:
        """
        Takes a user message, calls the LLM, and returns the response.
        """
        # YOUR CODE HERE
        pass


In [ ]:
# Solution
import uuid

class GoogleAgentSolution:
    def __init__(self, system_instruction: str, tools: list = None, enable_caching: bool = False):
        self.system_instruction = system_instruction
        self.tools = tools if tools else []
        self.history = []
        self.enable_caching = enable_caching
        
        if self.enable_caching:
            # In reality, this calls Google's API to upload and cache the tokens
            self.cache_id = f"cache_{uuid.uuid4().hex[:8]}"
            print(f"[System] Cached system instructions at {self.cache_id}")
            
    def chat(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        
        # Prepare payload
        payload = {
            "messages": self.history,
            "tools": [t.__name__ for t in self.tools] # Simplified
        }
        
        if self.enable_caching:
            payload["cached_content_id"] = self.cache_id
        else:
            payload["system_instruction"] = self.system_instruction
            
        # Mock API Call
        print(f"[API Call] Payload sent to Gemini: {payload.keys()}")
        
        response_text = f"I am a Gemini Agent responding to: {user_message}"
        self.history.append({"role": "model", "content": response_text})
        
        return response_text

agent = GoogleAgentSolution("You are a helpful assistant.", enable_caching=True)
print(agent.chat("Hello!"))


## 🔨 Exercise 2: Implementing JSON Schema Extraction

When you pass a Python function to an LLM as a "Tool", the ADK must convert the Python function signature into a JSON Schema that the LLM understands.

**Task**: Write a function `generate_tool_schema(func)` that takes a basic Python function and returns a dictionary representing its JSON schema (specifically the name, description, and properties).

In [ ]:
def get_weather(location: str, unit: str = "celsius") -> str:
    """Get the current weather in a given location."""
    pass

def generate_tool_schema(func) -> dict:
    """
    Should return a dict like:
    {
        "name": "get_weather",
        "description": "Get the current weather in a given location.",
        "parameters": {
            "location": "string",
            "unit": "string"
        }
    }
    """
    # YOUR CODE HERE
    pass

# print(generate_tool_schema(get_weather))

In [ ]:
# Solution
import inspect

def generate_tool_schema_solution(func) -> dict:
    schema = {
        "name": func.__name__,
        "description": inspect.getdoc(func) or "",
        "parameters": {}
    }
    
    sig = inspect.signature(func)
    for param_name, param in sig.parameters.items():
        # Basic type mapping (in reality, you'd handle complex types, enums, etc.)
        param_type = "string" if param.annotation == str else "object"
        schema["parameters"][param_name] = param_type
        
    return schema

print("Solution Output:")
import json
print(json.dumps(generate_tool_schema_solution(get_weather), indent=2))


## 📝 Day 31 Review

### Concepts Validated Today
- [ ] The architectural advantages of **Gemini** (Multimodality, Massive Context, Caching).
- [ ] How to abstract API payloads to utilize **Context Caching** efficiently.
- [ ] Using Python's `inspect` module to dynamically generate **JSON Schemas** from standard python functions for LLM Tool definitions.

### Tomorrow's Preview
**Day 32: Tool Use and Function Calling** — We dive deeper into Function Calling. How does the LLM actually invoke the tool, and how do we safely execute it and handle errors?